## Status 
- twitter business unit and aggregated services is currently calculated by using minnie's dataset (helper/tw_minnie_preBU.csv)
- also minnie's weekly gnl dataset is missing week 51 & 52? 
- 

In [32]:
platformID = 'TWI'

In [33]:
from IPython.display import display

import pandas as pd
pd.set_option('display.float_format', '{:.0f}'.format)

In [34]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config_GAM2025 import gam_info
import functions

In [35]:
# Load country mapping
country_map = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='CountryID')[['PlaceID', 'YT-_FBE_codes']]
# Load country mapping
week_map = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period')[['w/c', 'WeekNumber_finYear']]


In [36]:
# Utility functions
def load_excel(path):
    return pd.read_excel(path, engine='openpyxl')

def load_csv(path):
    return pd.read_csv(path)

def standardize_country_codes(df, column='Country Code'):
    return df.replace({column: {'WLF': 'WFI', '* Total': 'Total'}})

def run_comparison(original_df, new_df, column_mapping, key_columns, method='integer', threshold=0.0001):
    if method == 'integer':
        return compare_dataframes_integer(original_df, new_df, column_mapping, key_columns)
    elif method == 'percentage':
        return compare_dataframes_percentage(original_df, new_df, column_mapping, key_columns, threshold)
    else:
        raise ValueError("Unknown comparison method")

In [37]:
def compare_dataframes_integer(original_df, new_df, column_mapping, key_columns_new):
    """
    Compare two DataFrames and return rows that are missing or different.

    Parameters:
    - original_df: DataFrame from the original source
    - new_df: DataFrame from the new source
    - column_mapping: dict mapping original_df column names to new_df column names
    - key_columns_new: list of key columns using new_df naming

    Returns:
    - missing_from_new: rows in original_df not found in new_df
    - differing_rows: rows where key matches but mapped columns differ
    """

    # Rename original_df to match new_df column names
    original_df_renamed = original_df.rename(columns=column_mapping)

    # Ensure all required columns exist
    all_columns = list(column_mapping.values())
    original_subset = original_df_renamed[all_columns].copy()
    new_subset = new_df[all_columns].copy()

    # Round numeric columns to nearest integer
    for col in all_columns:
        if pd.api.types.is_numeric_dtype(original_subset[col]) and pd.api.types.is_numeric_dtype(new_subset[col]):
            original_subset[col] = original_subset[col].round(0).astype('Int64')
            new_subset[col] = new_subset[col].round(0).astype('Int64')
        
    # Merge to find differences
    merged = pd.merge(
        original_subset,
        new_subset,
        on=key_columns_new,
        how='outer',
        suffixes=('_orig', '_new'),
        indicator=True
    )

    # Missing rows: present in original but not in new
    missing_from_new = merged[merged['_merge'] == 'left_only']

    # Differing rows: same keys but different values
    comparison_cols = [col for col in all_columns if col not in key_columns_new]
        
    differing_rows = merged[
        (merged['_merge'] == 'both') &
        merged[[f"{col}_orig" for col in comparison_cols]].ne(
            merged[[f"{col}_new" for col in comparison_cols]].values
        ).any(axis=1)
    ]

    return missing_from_new, differing_rows

In [38]:
def compare_dataframes_percentage(original_df, new_df, column_mapping, key_columns_new, threshold=0.0001):
    """
    Compare two DataFrames and return rows that are missing or have percentage differences.

    Parameters:
    - original_df: DataFrame from the original source
    - new_df: DataFrame from the new source
    - column_mapping: dict mapping original_df column names to new_df column names
    - key_columns_new: list of key columns using new_df naming
    - threshold: minimum absolute difference to consider as significant

    Returns:
    - missing_from_new: rows in original_df not found in new_df
    - differing_rows: rows where key matches but mapped columns differ beyond threshold
    """

    # Rename original_df to match new_df column names
    original_df_renamed = original_df.rename(columns=column_mapping)

    # Ensure all required columns exist
    all_columns = list(column_mapping.values())
    original_subset = original_df_renamed[all_columns].copy()
    new_subset = new_df[all_columns].copy()

    # Merge to find differences
    merged = pd.merge(
        original_subset,
        new_subset,
        on=key_columns_new,
        how='outer',
        suffixes=('_orig', '_new'),
        indicator=True
    )

    # Missing rows: present in original but not in new
    missing_from_new = merged[merged['_merge'] == 'left_only']

    # Compute differences
    comparison_cols = [col for col in all_columns if col not in key_columns_new]
    for col in comparison_cols:
        merged[f"{col}_diff"] = merged[f"{col}_new"] - merged[f"{col}_orig"]

    # Filter rows where any difference exceeds threshold
    diff_mask = merged['_merge'] == 'both'
    for col in comparison_cols:
        diff_mask &= merged[f"{col}_diff"].abs() > threshold

    differing_rows = merged[diff_mask]

    return missing_from_new, differing_rows


In [39]:
# Dataset configuration
datasets = [
    
    {
        "name": "Engagements",
        "original_path": "../../../../Research Projects/GAM/Digital GAM/2025/Social Media/data/final data/TW_GAM2025_REDSHIFT.xlsx",
        "new_path": f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT.csv",
        "column_mapping": {
            "tw_account_id": "tw_account_id",
            "week_commencing": "w/c",
            "tweet_engagements": "tweet engagements"
        },
        "key_columns": ["tw_account_id", "w/c"],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": False
        },
        "notes": "262 accounts in minnie's dataset - I suspect she got all accounts the BBC has "
    },
    
    {
        "name": "pre Business Unites",
        "original_path": "../helper/tw_minnie_preBU.csv",
        "new_path": f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_uniqueViewer_country.csv",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'TW Account ID': 'Channel ID', 
            'Twitter Engaged Users by Country': 'uv_by_country'
        },
        "key_columns": ["Channel ID", "w/c", 'PlaceID', 'ServiceID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "notes": ""
    },
    {
        "name": "GNL Weekly",
        "original_path": "../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/WEEKLY Twitter GNL.xlsx",
        "new_path": "../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_GNLbyCountry.xlsx",
        "column_mapping": {
            "Service Code": "ServiceID",
            "Country Code": "PlaceID",
            "Twitter Engaged Users by Country": "Reach",
            "w/c": "w/c"
        },
        "key_columns": ["ServiceID", "PlaceID", "w/c"],
        "method": "integer",
        "preprocess": {
            "standardize_country": True,
            "week_mapping": True
        },
        "notes": """the missing are all from the last two weeks. 
        I don't see these values in the alteryx workflow either so not sure where they come from in 
        the output file. 
        differences: looking at the biggest difference I fidn the same value from my calculation in
        alteryx but the excel sheet has a different number... """
    },
]
'''{
        "name": "Country",
        "original_path": "../../../../Research Projects/GAM/Digital GAM/2024/Social Media/data/Final Raw/Twitter Engagements inc country.xlsx",
        "new_path": f'../data/raw/{platformID}/stale_Twitter Engagements inc country.xlsx',
        "column_mapping": {
            "TW Account ID": "tw_account_id",
            "TW Service Code": 'ServiceID', 
            "Country": "PlaceID",
            "Week Number": "Week Number",
            "Engagement %": "country_%"
        },
        "key_columns": ["tw_account_id", "w/c", "PlaceID", "ServiceID"],
        "method": "percentage",
        "threshold": 0.0001,
        "preprocess": {
            "standardize_country": False,
            "week_mapping": False
        },
        "notes": "has older data in that isn't of the current GAM year"
    },'''

'{\n        "name": "Country",\n        "original_path": "../../../../Research Projects/GAM/Digital GAM/2024/Social Media/data/Final Raw/Twitter Engagements inc country.xlsx",\n        "new_path": f\'../data/raw/{platformID}/stale_Twitter Engagements inc country.xlsx\',\n        "column_mapping": {\n            "TW Account ID": "tw_account_id",\n            "TW Service Code": \'ServiceID\', \n            "Country": "PlaceID",\n            "Week Number": "Week Number",\n            "Engagement %": "country_%"\n        },\n        "key_columns": ["tw_account_id", "w/c", "PlaceID", "ServiceID"],\n        "method": "percentage",\n        "threshold": 0.0001,\n        "preprocess": {\n            "standardize_country": False,\n            "week_mapping": False\n        },\n        "notes": "has older data in that isn\'t of the current GAM year"\n    },'

In [41]:
datasets = [
    {
        "name": "Twitter ALL Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter ALL.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_ALLbyCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter ANW Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter ANW.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_ANWbyCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter ANY Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter ANY.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_ANYbyCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter AX2 Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter AX2.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_AX2byCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter AXE Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter AXE.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_AXEbyCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter EN2 Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter EN2 by country.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_EN2byCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter ENG Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter ENG by country.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_ENGbyCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter ENW Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter ENW by country.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_ENWbyCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter FOA Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter FOA.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_FOAbyCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter GNL Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter GNL.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_GNLbyCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter MA- Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter MA.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_MA-byCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter TOT Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter TOT.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_TOTbyCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter WOR Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter WOR.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_WORbyCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter WSE Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter WSE.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_WSEbyCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
    {
        "name": "Twitter WSL Platform",
        "original_path": f"../../../../Research Projects/GAM/Digital GAM/2025/Social Media/Output/Weekly/Weekly Twitter WSL.xlsx",
        "new_path": f"../data/singlePlatform/TWI/weekly/GAM2025_WEEKLY_TWI_WSLbyCountry.xlsx",
        "column_mapping": {
            'w/c': 'w/c', 
            'Country Code': 'PlaceID', 
            'Service Code': 'ServiceID', 
            'Platform': 'PlatformID',
            'Reach': 'Reach',
        },
        "key_columns": ['w/c', 'PlaceID', 'ServiceID', 'PlatformID'],
        "method": "integer",
        "preprocess": {
            "standardize_country": False,
            "week_mapping": True
        },
        "comment": """  
        
        """
    },
]
# Execute comparisons
for ds in datasets:
    print(f"\n--- Processing {ds['name']} ---")

    orig = functions.load_excel(ds["original_path"]) if ds["original_path"].endswith(".xlsx") else functions.load_csv(ds["original_path"])
    new  = functions.load_excel(ds["new_path"]) if ds["new_path"].endswith(".xlsx") else functions.load_csv(ds["new_path"])

    if ds["name"] == "Facebook Factors input":
        new['fb_page_id'] = new['fb_page_id'].astype(str)
        orig = orig.dropna(subset=['page_consumptions', 'page_video_complete_views_30s_autoplayed'],
                           how='any')
        
    # Special preprocessing for Country Percentage dataset
    if ds["name"] == "Country Percentage":
        
        # Rename 'Country' to 'YouTube Codes' in original data and merge with mapping
        orig = orig.rename(columns={'Country': 'YouTube Codes'})
        orig = orig.merge(country_map, on='YouTube Codes', how='left').drop(columns=['YouTube Codes'])

    if "Country Code" in orig.columns:
        orig = functions.standardize_country_codes(orig)
    if "Country Code" in new.columns:
        new = functions.standardize_country_codes(new)

    # Rename columns according to mapping
    orig = orig.rename(columns={k: v for k, v in ds["column_mapping"].items() if k in orig.columns})
    new  = new.rename(columns={k: v for k, v in ds["column_mapping"].items() if k in new.columns})

    # Special preprocessing for Country Percentage dataset
    if ds['preprocess']['week_mapping']:
        # add w/c using Week Number
        orig = orig.merge(week_map, left_on='Week Number', right_on='WeekNumber_finYear',
                                              how='left').drop(columns=['Week Number', 'WeekNumber_finYear'])

    # Ensure 'w/c' columns are datetime in both DataFrames
    if 'w/c' in orig.columns:
        orig['w/c'] = pd.to_datetime(orig['w/c'], errors='coerce')
    if 'w/c' in new.columns:
        new['w/c'] = pd.to_datetime(new['w/c'], errors='coerce')

    missing, different = functions.run_comparison(
        orig, new,
        ds["column_mapping"],
        ds["key_columns"],
        method=ds.get("method", "integer"),
        threshold=ds.get("threshold", 0.0001)
    )

    print("Rows missing from new:")
    display(missing)
    print("Rows with differences:")
    if not different.empty:
        numeric_cols = [col for col in different.columns if col.endswith('_orig') and pd.api.types.is_numeric_dtype(different[col])]
    
        for col in numeric_cols:
            base_col = col.replace('_orig', '')
            new_col = f"{base_col}_new"
            diff_col = f"{base_col}_diff"
            if new_col in different.columns:
                different[diff_col] = different[col] - different[new_col]
        
            display(different.sort_values(by=[col for col in different.columns if col.endswith('_diff')][0], ascending=False))
        else:
            display(different)



--- Processing Twitter ALL Platform ---
Rows missing from new:


,w/c,PlaceID,ServiceID,PlatformID,Reach_orig,Reach_new,_merge
8,2024-04-01,BHZ,ALL,TWI,327,<NA>,left_only
19,2024-04-01,CRO,ALL,TWI,218,<NA>,left_only
56,2024-04-01,MLD,ALL,TWI,35,<NA>,left_only
58,2024-04-01,MTG,ALL,TWI,327,<NA>,left_only
59,2024-04-01,NAM,ALL,TWI,0,<NA>,left_only
...,...,...,...,...,...,...,...
5336,2025-03-24,BHZ,ALL,TWI,101,<NA>,left_only
5347,2025-03-24,CRO,ALL,TWI,67,<NA>,left_only
5384,2025-03-24,MLD,ALL,TWI,16,<NA>,left_only
5386,2025-03-24,MTG,ALL,TWI,101,<NA>,left_only


Rows with differences:


,w/c,PlaceID,ServiceID,PlatformID,Reach_orig,Reach_new,_merge,Reach_diff
1280,2024-06-17,USA,ALL,TWI,5800975,80468,both,5720507
1386,2024-06-24,USA,ALL,TWI,4727797,45604,both,4682193
1279,2024-06-17,USA,ALL,TWI,5800975,1395513,both,4405462
1810,2024-07-22,USA,ALL,TWI,4209366,50944,both,4158422
1385,2024-06-24,USA,ALL,TWI,4727797,761850,both,3965947
...,...,...,...,...,...,...,...,...
2228,2024-08-19,UK,ALL,TWI,263516,1970504,both,-1706988
525,2024-04-29,UK,ALL,TWI,214514,2000412,both,-1785898
1169,2024-06-10,UK,ALL,TWI,139578,2246654,both,-2107076
2850,2024-09-30,USA,ALL,TWI,804967,3207133,both,-2402166


,w/c,PlaceID,ServiceID,PlatformID,Reach_orig,Reach_new,_merge,Reach_diff
0,2024-04-01,AFG,ALL,TWI,14796,16755,both,-1959
1,2024-04-01,ARG,ALL,TWI,3112,11513,both,-8401
2,2024-04-01,AUS,ALL,TWI,389161,152934,both,236227
3,2024-04-01,AUS,ALL,TWI,389161,336782,both,52379
4,2024-04-01,AZE,ALL,TWI,15,48,both,-33
...,...,...,...,...,...,...,...,...
5426,2025-03-24,USA,ALL,TWI,464997,45577,both,419420
5427,2025-03-24,UZB,ALL,TWI,30,0,both,30
5428,2025-03-24,VEN,ALL,TWI,5076,0,both,5076
5429,2025-03-24,YEM,ALL,TWI,253,0,both,253


...


--- Processing Twitter WSL Platform ---
Rows missing from new:


,w/c,PlaceID,ServiceID,PlatformID,Reach_orig,Reach_new,_merge
153,2024-04-01,PAR,SPA,TWI,0,<NA>,left_only
5137,2024-07-22,PAR,SPA,TWI,0,<NA>,left_only
5449,2024-07-29,PAR,SPA,TWI,0,<NA>,left_only
5761,2024-08-05,PAR,SPA,TWI,0,<NA>,left_only


Rows with differences:


,w/c,PlaceID,ServiceID,PlatformID,Reach_orig,Reach_new,_merge,Reach_diff
16096,2025-03-24,TUR,TUR,TWI,1122939,0,both,1122939
14414,2025-02-17,INO,INO,TWI,837329,33863,both,803466
15658,2025-03-17,INO,INO,TWI,299951,0,both,299951
15970,2025-03-24,INO,INO,TWI,274979,0,both,274979
2890,2024-06-03,IND,HIN,TWI,480775,211714,both,269061
...,...,...,...,...,...,...,...,...
9990,2024-11-11,BRA,POR,TWI,39674,668470,both,-628796
2777,2024-05-27,USA,MAN,TWI,45066,733354,both,-688288
13162,2025-01-20,IND,HIN,TWI,154953,1183397,both,-1028444
6320,2024-08-19,IND,HIN,TWI,103178,1161768,both,-1058590


,w/c,PlaceID,ServiceID,PlatformID,Reach_orig,Reach_new,_merge,Reach_diff
0,2024-04-01,AFG,DAR,TWI,590,282,both,308
1,2024-04-01,AFG,PAS,TWI,4811,3025,both,1786
2,2024-04-01,AFG,PER,TWI,9396,13447,both,-4051
3,2024-04-01,ARG,POR,TWI,585,1384,both,-799
4,2024-04-01,ARG,SPA,TWI,2477,9709,both,-7232
...,...,...,...,...,...,...,...,...
16179,2025-03-24,USA,UZB,TWI,2,0,both,2
16180,2025-03-24,UZB,UZB,TWI,30,0,both,30
16181,2025-03-24,VEN,SPA,TWI,5076,0,both,5076
16182,2025-03-24,YEM,ARA,TWI,253,0,both,253
